## CAEEG 三分类数据集构建

本脚本用于构建三分类数据集：
- 使用 Assess-01 中 `resting state befor` 范式中的数据
- 每个受试者使用 `closed_eyes_epochs.npy` 和 `open_eyes_clean_data.npy` 数据
- 标签从 `CAEEG Label.xlsx` 文件中读取：NC（标签0），MCI（标签1），AD（标签2）
- 使用"标签 1"列对应Assess-01的数据
- 对长序列进行划窗处理，窗口长度和重叠率可配置
- 输出三个数据集：EC（闭眼）、EO（睁眼）、All（合并）


In [10]:
import os
import numpy as np
import pandas as pd
from pathlib import Path


## 参数设置和路径配置


In [11]:
# ==================== 参数设置 ====================
# 划窗参数
window_time = 180.0  # 窗口时间（秒），例如1s
overlap_ratio = 0.0  # 窗口重叠率，例如0.5表示50%
sampling_rate = 128  # 采样频率（Hz）
n_channels = 20  # 通道数
USE_QC = False  # 是否对窗口做质量检测；False 时保留全部窗口

# 计算窗口长度（样本数）
window_length = int(window_time * sampling_rate)  # 例如1s * 128Hz = 128个样本
step_length = int(window_length * (1 - overlap_ratio))  # 步长

# 是否启用数据增强（Random Cropping 双路）；True 时保存形状为 (2, N, WINDOW_SIZE, n_channels)
# 其中 array[0] 为原始切片，array[1] 为增强切片，同一 N 下标严格对齐
USE_AUGMENTATION = False
AUGMENTATION_RANDOM_SEED = 42

print(f"划窗参数:")
print(f"  窗口时间: {window_time}s")
print(f"  窗口长度: {window_length} 个样本")
print(f"  重叠率: {overlap_ratio*100}%")
print(f"  步长: {step_length} 个样本")
print(f"  采样频率: {sampling_rate} Hz")
print(f"  通道数: {n_channels}")
print(f"  窗口质量检测: {'开启' if USE_QC else '关闭'}")

# ==================== 路径设置 ====================
# 输入数据路径
input_paradigm_dir = r"/root/autodl-tmp/CAEEG 128Hz/Assess-01/resting state befor"

# 标签文件路径
class_file_path = r"/root/autodl-tmp/CAEEG 128Hz/CAEEG Label.xlsx"

# 输出数据路径（基础路径）
output_base_dir = r"/root/autodl-tmp/ADMamba/dataset"
# 构建输出目录名称：CAEEG_{window_time}s_{overlap_ratio}
output_dir_name = f"CAEEG3_{window_time}s_{overlap_ratio}"

# 直接创建三个输出目录：EC（闭眼）、EO（睁眼）、All（合并）
if USE_AUGMENTATION:
    output_ec_dir = os.path.join(output_base_dir, f"{output_dir_name}_EC_Aug")
    output_eo_dir = os.path.join(output_base_dir, f"{output_dir_name}_EO_Aug")
    output_all_dir = os.path.join(output_base_dir, f"{output_dir_name}_All_Aug")
else:
    output_ec_dir = os.path.join(output_base_dir, f"{output_dir_name}_EC")
    output_eo_dir = os.path.join(output_base_dir, f"{output_dir_name}_EO")
    output_all_dir = os.path.join(output_base_dir, f"{output_dir_name}_All")

# 每个目录下创建Feature和Label子目录
for output_dir in [output_ec_dir, output_eo_dir, output_all_dir]:
    os.makedirs(os.path.join(output_dir, "Feature"), exist_ok=True)
    os.makedirs(os.path.join(output_dir, "Label"), exist_ok=True)

print(f"\n路径设置:")
print(f"  输入数据路径: {input_paradigm_dir}")
print(f"  标签文件路径: {class_file_path}")
print(f"  输出基础路径: {output_base_dir}")
print(f"  EC输出路径: {output_ec_dir}")
print(f"  EO输出路径: {output_eo_dir}")
print(f"  All输出路径: {output_all_dir}")


划窗参数:
  窗口时间: 180.0s
  窗口长度: 23040 个样本
  重叠率: 0.0%
  步长: 23040 个样本
  采样频率: 128 Hz
  通道数: 20
  窗口质量检测: 关闭

路径设置:
  输入数据路径: /root/autodl-tmp/CAEEG 128Hz/Assess-01/resting state befor
  标签文件路径: /root/autodl-tmp/CAEEG 128Hz/CAEEG Label.xlsx
  输出基础路径: /root/autodl-tmp/ADMamba/dataset
  EC输出路径: /root/autodl-tmp/ADMamba/dataset/CAEEG3_180.0s_0.0_EC
  EO输出路径: /root/autodl-tmp/ADMamba/dataset/CAEEG3_180.0s_0.0_EO
  All输出路径: /root/autodl-tmp/ADMamba/dataset/CAEEG3_180.0s_0.0_All


## 读取标签文件


In [12]:
# 读取CAEEG Label.xlsx文件
# Excel文件结构：列头为[英文名, 标签 1]，第一行是表头
# 使用"标签 1"列对应Assess-01的数据
df_labels = None

if os.path.exists(class_file_path):
    # 读取Excel文件，第一行作为表头
    df_labels = pd.read_excel(class_file_path, header=0)
    print(f"标签文件列名: {df_labels.columns.tolist()}")
    print(f"\n标签文件前5行:")
    print(df_labels.head())
    print(f"\n标签文件形状: {df_labels.shape}")
    
    # 确认文件包含所需的列
    if "英文名" not in df_labels.columns:
        print(f"错误: Excel文件中找不到'英文名'列")
        print(f"可用列: {df_labels.columns.tolist()}")
    elif "标签 1" not in df_labels.columns:
        print(f"错误: Excel文件中找不到'标签 1'列")
        print(f"可用列: {df_labels.columns.tolist()}")
    else:
        name_col = "英文名"
        class_col = "标签 1"  # 使用"标签 1"列对应Assess-01
        print(f"\n姓名列: {name_col}")
        print(f"标签列（用于Assess-01）: {class_col}")
else:
    print(f"错误: 找不到标签文件 {class_file_path}")


标签文件列名: ['编号', '英文名', '性别', '年龄', '文化程度', 'Unnamed: 5', '标签 1', 'MoCA 1', 'MMSE 1', 'Unnamed: 9', '标签 2', 'MoCA 2', 'MMSE 2']

标签文件前5行:
   编号           英文名 性别  年龄    文化程度  Unnamed: 5 标签 1  MoCA 1  MMSE 1  \
0   1  tianxiaolong  女  78      本科         NaN  MCI      23      28   
1   2       wangyan  男  78  研究生及以上         NaN  MCI      25      29   
2   3   yuandonghui  女  80      高中         NaN   NC      29      30   
3   4     wanglijia  男  85      本科         NaN  MCI      24      26   
4   5       wangjun  女  82      中专         NaN   NC      27      26   

   Unnamed: 9 标签 2  MoCA 2  MMSE 2  
0         NaN   NC    28.0    28.0  
1         NaN   NC    27.0    29.0  
2         NaN   NC    30.0    30.0  
3         NaN   NC    27.0    26.0  
4         NaN  NaN     NaN     NaN  

标签文件形状: (111, 13)

姓名列: 英文名
标签列（用于Assess-01）: 标签 1


## 划窗处理函数


In [13]:
def sliding_window(data, window_length, step_length):
    """
    对数据进行划窗处理
    
    参数:
        data: numpy数组，形状为 (time_points, channels) 或 (time_points,)
        window_length: 窗口长度（样本数）
        step_length: 步长（样本数）
    
    返回:
        windows: numpy数组，形状为 (n_windows, window_length, channels)
    """
    # 确保数据是2D的
    if data.ndim == 1:
        data = data.reshape(-1, 1)
    elif data.ndim == 2:
        pass  # 已经是2D
    else:
        raise ValueError(f"数据维度应为1D或2D，但得到 {data.ndim}D")
    
    time_points, channels = data.shape
    
    # 计算可以完整划窗的数量
    n_full_windows = (time_points - window_length) // step_length + 1
    
    # 计算剩余长度
    remaining = time_points - (n_full_windows - 1) * step_length - window_length
    
    # 如果剩余长度不足一个窗口，从中间位置开始划窗
    if remaining > 0 and remaining < window_length:
        start_idx = remaining // 2
    else:
        start_idx = 0
    
    # 生成窗口
    windows = []
    for i in range(n_full_windows):
        begin = start_idx + i * step_length
        end = begin + window_length
        if end <= time_points:
            window = data[begin:end, :]
            windows.append(window)
    
    if len(windows) == 0:
        # 如果无法生成窗口，返回空数组
        return np.array([]).reshape(0, window_length, channels)
    
    windows = np.array(windows)
    
    # 确保形状为 (n_windows, window_length, channels)
    return windows


def segment_random_crop_pair_array(
    data,
    window_length,
    step_length,
    sfreq,
    n_channels,
    enable_qc=True,
    rng=None,
    random_seed=42,
    verbose=False,
):
    """
    Random Cropping 双路成对增强：生成两条窗口序列（原始/增强），并保证窗口一一对应。

    返回:
        (windows_ori, windows_aug, meta)
        - windows_ori/windows_aug: shape=(N, window_length, n_channels)
        - meta: 包含 start_ori/start_aug/gap/n_windows/stride/L_work 等信息

    说明:
        - 两路使用相同 stride (=step_length) 与相同 n_windows
        - QC 时按索引成对过滤：仅当同一下标两窗均通过才保留
    """
    if rng is None:
        rng = np.random.default_rng(random_seed)

    if data.ndim == 1:
        data = data.reshape(-1, 1)
    elif data.ndim != 2:
        raise ValueError(f"数据维度应为1D或2D，但得到 {data.ndim}D")

    T, C = data.shape
    W = int(window_length)
    stride = int(step_length)
    stride = max(1, stride)

    empty_meta = {
        "start_ori": None,
        "start_aug": None,
        "gap": None,
        "n_windows": 0,
        "reserved_tail": None,
        "stride": stride,
        "L_work": None,
    }

    if T < W or C != int(n_channels):
        return np.empty((0, W, C)), np.empty((0, W, C)), empty_meta

    # 末尾预留：避免随机起点导致越界
    last_start = ((T - W) // stride) * stride
    tail = T - (last_start + W)

    if tail == 0:
        reserved_tail = W
    elif tail < (W / 2.0):
        reserved_tail = 2 * W
    else:
        reserved_tail = W

    L_work = T - reserved_tail
    if L_work < W:
        return np.empty((0, W, C)), np.empty((0, W, C)), {**empty_meta, "reserved_tail": reserved_tail, "L_work": L_work}

    n_windows = (L_work - W) // stride + 1
    max_start = L_work - W - (n_windows - 1) * stride
    if max_start < 0:
        return np.empty((0, W, C)), np.empty((0, W, C)), {**empty_meta, "reserved_tail": reserved_tail, "L_work": L_work}

    d_min = int(np.ceil(W / 10.0))
    d_max = int(np.floor(W / 2.0))
    if d_min > d_max:
        raise ValueError(
            f"window_length={W} 过小：无法满足起点间隔 ∈ [W/10, W/2]（得到 d_min={d_min}, d_max={d_max}）"
        )

    if max_start == 0:
        start_ori = 0
        start_aug = 0
        gap = 0
    else:
        d_lo, d_hi = d_min, d_max
        if max_start < d_min:
            d_lo = 1
            d_hi = min(d_max, max_start)
        if d_lo > d_hi:
            start_ori = 0
            start_aug = 0
            gap = 0
        else:
            start_ori = None
            start_aug = None
            gap = None
            for _ in range(500):
                g = int(rng.integers(d_lo, d_hi + 1))
                upward = rng.random() < 0.5
                if upward:
                    lo, hi = 0, max_start - g
                    if lo > hi:
                        continue
                    start_ori = int(rng.integers(lo, hi + 1))
                    start_aug = start_ori + g
                else:
                    lo, hi = g, max_start
                    if lo > hi:
                        continue
                    start_ori = int(rng.integers(lo, hi + 1))
                    start_aug = start_ori - g
                gap = g
                break

            if start_ori is None:
                raise RuntimeError(
                    "无法在约束下采样到合法的 (start_ori, start_aug)；请检查 window_length、step_length 与数据长度。"
                )

    windows_ori = np.empty((n_windows, W, C), dtype=data.dtype)
    windows_aug = np.empty((n_windows, W, C), dtype=data.dtype)
    for k in range(n_windows):
        i_ori = start_ori + k * stride
        i_aug = start_aug + k * stride
        windows_ori[k] = data[i_ori : i_ori + W, :]
        windows_aug[k] = data[i_aug : i_aug + W, :]

    meta = {
        "start_ori": int(start_ori),
        "start_aug": int(start_aug),
        "gap": int(gap),
        "n_windows": int(n_windows),
        "reserved_tail": int(reserved_tail),
        "L_work": int(L_work),
        "stride": int(stride),
    }

    if enable_qc and n_windows > 0:
        qc_ori = window_qc(windows_ori, sfreq, n_channels, verbose=verbose)
        qc_aug = window_qc(windows_aug, sfreq, n_channels, verbose=verbose)
        qc_pair = qc_ori & qc_aug
        if verbose:
            print(
                f"  [random_crop_pair] QC: 总窗口 {n_windows}, 两路均通过 {qc_pair.sum()}, 丢弃 {n_windows - qc_pair.sum()}"
            )
        windows_ori = windows_ori[qc_pair]
        windows_aug = windows_aug[qc_pair]
        meta["qc_pass_ori"] = qc_ori
        meta["qc_pass_aug"] = qc_aug
        meta["qc_pass_pair"] = qc_pair

    return windows_ori, windows_aug, meta


def window_qc(
    windows,
    sfreq,
    n_channels,
    bad_channel_ratio_thresh=0.2,  # >20% 通道坏 → 丢窗
    abs_amp_thresh=200.0,          # µV，保护性绝对阈值
    verbose=False
):
    """
    对窗口进行质量检测，识别异常窗口。

    参数:
    - windows (np.ndarray): 窗口数据，3D数组，形状为 (n_windows, window_length, channels) 或 (n_windows, channels, window_length)
    - sfreq (float): 采样率 (Hz)
    - n_channels (int): 通道数，用于判断和修正维度顺序
    - bad_channel_ratio_thresh (float): 坏通道比例阈值，超过此比例的窗口将被标记为异常。默认: 0.2
    - abs_amp_thresh (float): 绝对幅值阈值 (µV)。默认: 200.0
    - verbose (bool): 是否打印详细信息。默认: False

    返回:
    - qc_pass (np.ndarray): 布尔数组，形状为 (n_windows,)，True 表示通过 QC
    """
    # 验证输入是3D数组
    if windows.ndim != 3:
        raise ValueError(f"windows 必须是 3D 数组，当前形状: {windows.shape}")

    original_shape = windows.shape

    # 如果最后一个维度是通道数，交换后两个维度
    if windows.shape[-1] == n_channels:
        windows = np.transpose(windows, (0, 2, 1))
        if verbose:
            print(f"  形状修正: {original_shape} -> {windows.shape} (交换后两个维度)")

    n_w, n_ch, n_t = windows.shape

    # ---------- A. 峰峰值 ----------
    ptp = np.ptp(windows, axis=-1)        # (n_w, n_ch)
    ptp_med = np.median(ptp, axis=0)
    ptp_mad = np.median(np.abs(ptp - ptp_med), axis=0) + 1e-6
    ptp_bad = ptp > (ptp_med + 6 * ptp_mad)

    # 绝对幅值保护
    absmax = np.max(np.abs(windows), axis=-1)
    abs_bad = absmax > abs_amp_thresh

    # ---------- B. 汇总坏道 ----------
    bad_channel_mask = ptp_bad | abs_bad
    bad_ratio = bad_channel_mask.sum(axis=1) / n_ch
    if verbose:
        print(f"  坏道数量: {bad_channel_mask.sum(axis=1)}")

    qc_pass = bad_ratio <= bad_channel_ratio_thresh

    return qc_pass


# 定义标签映射：NC（0），MCI（1），AD（2）
def get_label(class_value):
    """根据CAEEG Label.xlsx中的值返回标签：NC->0, MCI->1, AD->2"""
    if pd.isna(class_value):
        return None
    class_str = str(class_value).strip().upper()
    if class_str == 'NC':
        return 0  # 正常人
    elif class_str == 'MCI':
        return 1  # MCI患者
    elif class_str == 'AD':
        return 2  # AD患者
    else:
        return None

# 处理数据
# 为三个数据集分别存储数据，按受试者编号组织
# 结构：{dataset_type: {subject_number: {'features': windows, 'label': label, 'subject_name': name}}}
all_data = {
    'EC': {},
    'EO': {},
    'All': {}
}

missing_subjects = []  # 记录找不到数据的受试者
missing_labels = []  # 记录找不到标签的受试者

# 构建受试者姓名到（标签, 编号）的映射（从Excel文件中）
subject_to_info = {}  # {subject_name: (label, number)}
if df_labels is not None and "英文名" in df_labels.columns and "标签 1" in df_labels.columns:
    name_col = "英文名"
    class_col = "标签 1"  # 使用"标签 1"列对应Assess-01
    number_col = "编号"  # 编号列
    
    if number_col not in df_labels.columns:
        print("错误: Excel文件缺少必需的列（需要'编号'列）")
    else:
        for idx, row in df_labels.iterrows():
            subject_name = str(row[name_col]).strip()
            class_value = row[class_col]
            number_value = row[number_col]
            label = get_label(class_value)
            if label is not None:
                # 将编号转换为整数
                try:
                    subject_number = int(number_value)
                    subject_to_info[subject_name] = (label, subject_number)
                except (ValueError, TypeError):
                    print(f"警告: 受试者 {subject_name} 的编号 '{number_value}' 无法转换为整数，跳过")
        # 注意：不打印每条记录，避免输出过多
        print(f"\n从标签文件中读取到 {len(subject_to_info)} 个受试者的标签信息")
        print(f"  正常人 (NC, 0): {sum(1 for lbl, num in subject_to_info.values() if lbl == 0)} 个")
        print(f"  MCI患者 (MCI, 1): {sum(1 for lbl, num in subject_to_info.values() if lbl == 1)} 个")
        print(f"  AD患者 (AD, 2): {sum(1 for lbl, num in subject_to_info.values() if lbl == 2)} 个")
elif df_labels is not None:
    print("错误: Excel文件缺少必需的列（需要'英文名'和'标签 1'列）")

# 处理输入目录
if not os.path.exists(input_paradigm_dir):
    print(f"错误: 输入数据路径不存在: {input_paradigm_dir}")
else:
    print(f"\n开始处理数据...")
    
    # 收集所有受试者信息
    all_subjects = []
    if os.path.exists(input_paradigm_dir):
        for item in os.listdir(input_paradigm_dir):
            subject_path = os.path.join(input_paradigm_dir, item)
            if os.path.isdir(subject_path):
                all_subjects.append((item, subject_path))
    
    print(f"找到 {len(all_subjects)} 个受试者文件夹")
    
    # 处理每个受试者
    for subject_name, subject_path in all_subjects:
        # 获取标签和编号
        label = None
        subject_number = None
        # 尝试多种方式匹配受试者姓名
        if subject_name in subject_to_info:
            label, subject_number = subject_to_info[subject_name]
        else:
            # 尝试不区分大小写匹配
            for name, (lbl, num) in subject_to_info.items():
                if name.lower() == subject_name.lower():
                    label = lbl
                    subject_number = num
                    break
        
        if label is None or subject_number is None:
            missing_labels.append(subject_name)
            print(f"  警告: 受试者 {subject_name} 在标签文件中找不到对应的标签，跳过")
            continue
        
        # 处理EC数据（closed_eyes_epochs.npy）
        ec_data_path = os.path.join(subject_path, "closed_eyes_clean_data.npy")
        ec_windows = None
        if os.path.exists(ec_data_path):
            try:
                ec_data = np.load(ec_data_path)
                # 如果数据是3D的 (epochs, time, channels)，需要reshape为2D
                if ec_data.ndim == 3:
                    # 检查最后一个维度是否是通道数
                    if ec_data.shape[-1] == n_channels:
                        # (epochs, time, channels) -> (total_time, channels)
                        ec_data = ec_data.reshape(-1, ec_data.shape[-1])
                    elif ec_data.shape[1] == n_channels:
                        # (epochs, channels, time) -> 转置为 (epochs, time, channels) -> (total_time, channels)
                        ec_data = ec_data.transpose(0, 2, 1).reshape(-1, n_channels)
                    else:
                        raise ValueError(f"EC数据3D形状 {ec_data.shape} 无法匹配通道数 {n_channels}")
                elif ec_data.ndim == 2:
                    # 检查最后一个维度是否是通道数
                    if ec_data.shape[-1] != n_channels:
                        # 如果第一个维度是通道数，则转置
                        if ec_data.shape[0] == n_channels:
                            ec_data = ec_data.T
                        else:
                            raise ValueError(f"EC数据2D形状 {ec_data.shape} 无法匹配通道数 {n_channels}")
                else:
                    raise ValueError(f"EC数据维度应为2D或3D，但得到 {ec_data.ndim}D")
                
                # 再次确保通道数正确
                if ec_data.shape[-1] != n_channels:
                    print(f"  警告: 受试者 {subject_name} 的EC数据通道数为 {ec_data.shape[-1]}，期望 {n_channels}，跳过")
                else:
                    # 进行划窗处理
                    if USE_AUGMENTATION:
                        ec_ori, ec_aug, _meta = segment_random_crop_pair_array(
                            ec_data,
                            window_length=window_length,
                            step_length=step_length,
                            sfreq=sampling_rate,
                            n_channels=n_channels,
                            enable_qc=USE_QC,
                            random_seed=AUGMENTATION_RANDOM_SEED,
                            verbose=False,
                        )
                        if len(ec_ori) > 0:
                            ec_windows = np.stack([ec_ori, ec_aug], axis=0)  # (2, N, W, C)
                        else:
                            ec_windows = np.empty((2, 0, window_length, n_channels))
                    else:
                        ec_windows = sliding_window(ec_data, window_length, step_length)
                        # 窗口质量检测，过滤异常窗口
                        if USE_QC and len(ec_windows) > 0:
                            qc_pass_ec = window_qc(ec_windows, sampling_rate, n_channels, verbose=False)
                            ec_windows = ec_windows[qc_pass_ec]
                    ec_n = ec_windows.shape[1] if USE_AUGMENTATION else len(ec_windows)
                    if ec_n > 0:
                        # 按受试者编号存储
                        if subject_number not in all_data['EC']:
                            all_data['EC'][subject_number] = {
                                'features': ec_windows,
                                'label': label,
                                'subject_name': subject_name
                            }
                        else:
                            # 如果该受试者已有数据，合并窗口
                            all_data['EC'][subject_number]['features'] = np.concatenate(
                                [all_data['EC'][subject_number]['features'], ec_windows],
                                axis=1 if USE_AUGMENTATION else 0,
                            )
            except Exception as e:
                print(f"  错误: 处理受试者 {subject_name} 的EC数据时出错: {e}")
        
        # 处理EO数据（open_eyes_clean_data.npy）
        eo_data_path = os.path.join(subject_path, "open_eyes_clean_data.npy")
        eo_windows = None
        if os.path.exists(eo_data_path):
            try:
                eo_data = np.load(eo_data_path)
                # 如果数据是3D的 (epochs, time, channels)，需要reshape为2D
                if eo_data.ndim == 3:
                    # 检查最后一个维度是否是通道数
                    if eo_data.shape[-1] == n_channels:
                        # (epochs, time, channels) -> (total_time, channels)
                        eo_data = eo_data.reshape(-1, eo_data.shape[-1])
                    elif eo_data.shape[1] == n_channels:
                        # (epochs, channels, time) -> 转置为 (epochs, time, channels) -> (total_time, channels)
                        eo_data = eo_data.transpose(0, 2, 1).reshape(-1, n_channels)
                    else:
                        raise ValueError(f"EO数据3D形状 {eo_data.shape} 无法匹配通道数 {n_channels}")
                elif eo_data.ndim == 2:
                    # 检查最后一个维度是否是通道数
                    if eo_data.shape[-1] != n_channels:
                        # 如果第一个维度是通道数，则转置
                        if eo_data.shape[0] == n_channels:
                            eo_data = eo_data.T
                        else:
                            raise ValueError(f"EO数据2D形状 {eo_data.shape} 无法匹配通道数 {n_channels}")
                else:
                    raise ValueError(f"EO数据维度应为2D或3D，但得到 {eo_data.ndim}D")
                
                # 再次确保通道数正确
                if eo_data.shape[-1] != n_channels:
                    print(f"  警告: 受试者 {subject_name} 的EO数据通道数为 {eo_data.shape[-1]}，期望 {n_channels}，跳过")
                else:
                    # 进行划窗处理
                    if USE_AUGMENTATION:
                        eo_ori, eo_aug, _meta = segment_random_crop_pair_array(
                            eo_data,
                            window_length=window_length,
                            step_length=step_length,
                            sfreq=sampling_rate,
                            n_channels=n_channels,
                            enable_qc=USE_QC,
                            random_seed=AUGMENTATION_RANDOM_SEED,
                            verbose=False,
                        )
                        if len(eo_ori) > 0:
                            eo_windows = np.stack([eo_ori, eo_aug], axis=0)  # (2, N, W, C)
                        else:
                            eo_windows = np.empty((2, 0, window_length, n_channels))
                    else:
                        eo_windows = sliding_window(eo_data, window_length, step_length)
                        # 窗口质量检测，过滤异常窗口
                        if USE_QC and len(eo_windows) > 0:
                            qc_pass_eo = window_qc(eo_windows, sampling_rate, n_channels, verbose=False)
                            eo_windows = eo_windows[qc_pass_eo]
                    eo_n = eo_windows.shape[1] if USE_AUGMENTATION else len(eo_windows)
                    if eo_n > 0:
                        # 按受试者编号存储
                        if subject_number not in all_data['EO']:
                            all_data['EO'][subject_number] = {
                                'features': eo_windows,
                                'label': label,
                                'subject_name': subject_name
                            }
                        else:
                            # 如果该受试者已有数据，合并窗口
                            all_data['EO'][subject_number]['features'] = np.concatenate(
                                [all_data['EO'][subject_number]['features'], eo_windows],
                                axis=1 if USE_AUGMENTATION else 0,
                            )
            except Exception as e:
                print(f"  错误: 处理受试者 {subject_name} 的EO数据时出错: {e}")
        
        # 处理All数据（合并EC和EO）
        ec_n = (ec_windows.shape[1] if USE_AUGMENTATION else len(ec_windows)) if ec_windows is not None else 0
        eo_n = (eo_windows.shape[1] if USE_AUGMENTATION else len(eo_windows)) if eo_windows is not None else 0
        if ec_n > 0 and eo_n > 0:
            try:
                # 合并EC和EO的窗口
                all_windows = np.concatenate([ec_windows, eo_windows], axis=1 if USE_AUGMENTATION else 0)
                # 按受试者编号存储
                if subject_number not in all_data['All']:
                    all_data['All'][subject_number] = {
                        'features': all_windows,
                        'label': label,
                        'subject_name': subject_name
                    }
                else:
                    # 如果该受试者已有数据，合并窗口
                    all_data['All'][subject_number]['features'] = np.concatenate(
                        [all_data['All'][subject_number]['features'], all_windows],
                        axis=1 if USE_AUGMENTATION else 0,
                    )
            except Exception as e:
                print(f"  错误: 处理受试者 {subject_name} 的All数据时出错: {e}")
        
        # 打印进度
        if ec_windows is not None or eo_windows is not None:
            def _n_windows(x):
                if x is None:
                    return 0
                if USE_AUGMENTATION:
                    return int(x.shape[1]) if x.ndim >= 2 else 0
                return int(len(x))

            ec_n = _n_windows(ec_windows)
            eo_n = _n_windows(eo_windows)
            ec_info = f"EC: {ec_n}窗口" if ec_n > 0 else "EC: 无"
            eo_info = f"EO: {eo_n}窗口" if eo_n > 0 else "EO: 无"
            print(f"  受试者 {subject_name} (编号: {subject_number}, 标签: {label}): {ec_info}, {eo_info}")
    
    print(f"\n处理完成!")
    for dataset_type in ['EC', 'EO', 'All']:
        n_subjects = len(all_data[dataset_type])
        def _count_windows(feat):
            if USE_AUGMENTATION:
                return int(feat.shape[1]) if isinstance(feat, np.ndarray) and feat.ndim >= 2 else 0
            return int(feat.shape[0]) if isinstance(feat, np.ndarray) else int(len(feat))

        total_windows = sum(_count_windows(d['features']) for d in all_data[dataset_type].values())
        print(f"  {dataset_type}数据集: {n_subjects} 个受试者, {total_windows} 个窗口样本")
        if n_subjects > 0:
            n_class0 = sum(1 for data in all_data[dataset_type].values() if data['label'] == 0)
            n_class1 = sum(1 for data in all_data[dataset_type].values() if data['label'] == 1)
            n_class2 = sum(1 for data in all_data[dataset_type].values() if data['label'] == 2)
            windows_class0 = sum(_count_windows(d['features']) for d in all_data[dataset_type].values() if d['label'] == 0)
            windows_class1 = sum(_count_windows(d['features']) for d in all_data[dataset_type].values() if d['label'] == 1)
            windows_class2 = sum(_count_windows(d['features']) for d in all_data[dataset_type].values() if d['label'] == 2)
            print(f"    正常人 (NC, 0): {n_class0} 个受试者, {windows_class0} 个窗口")
            print(f"    MCI患者 (MCI, 1): {n_class1} 个受试者, {windows_class1} 个窗口")
            print(f"    AD患者 (AD, 2): {n_class2} 个受试者, {windows_class2} 个窗口")
    
    if missing_labels:
        print(f"  缺少标签: {len(missing_labels)} 个受试者 - {missing_labels[:10]}{'...' if len(missing_labels) > 10 else ''}")



从标签文件中读取到 111 个受试者的标签信息
  正常人 (NC, 0): 56 个
  MCI患者 (MCI, 1): 43 个
  AD患者 (AD, 2): 12 个

开始处理数据...
找到 116 个受试者文件夹
  受试者 baijin (编号: 108, 标签: 0): EC: 1窗口, EO: 1窗口
  受试者 chaizhaoji (编号: 46, 标签: 1): EC: 1窗口, EO: 1窗口
  受试者 chenfukang (编号: 82, 标签: 1): EC: 1窗口, EO: 1窗口
  受试者 congshuren (编号: 22, 标签: 1): EC: 1窗口, EO: 1窗口
  受试者 dengqing (编号: 13, 标签: 0): EC: 1窗口, EO: 1窗口
  受试者 dingyouxiang (编号: 32, 标签: 1): EC: 1窗口, EO: 1窗口
  受试者 dujiansun (编号: 62, 标签: 2): EC: 1窗口, EO: 1窗口
  受试者 dujingxian (编号: 15, 标签: 2): EC: 1窗口, EO: 1窗口
  受试者 fenglihui (编号: 64, 标签: 0): EC: 1窗口, EO: 1窗口
  受试者 fengyeheng (编号: 23, 标签: 0): EC: 1窗口, EO: 1窗口
  受试者 gaomeizhu (编号: 109, 标签: 0): EC: 1窗口, EO: 1窗口
  受试者 gaotonghuai (编号: 59, 标签: 2): EC: 1窗口, EO: 1窗口
  受试者 gedeyun (编号: 10, 标签: 2): EC: 1窗口, EO: 1窗口
  受试者 gujinfen (编号: 33, 标签: 1): EC: 1窗口, EO: 1窗口
  受试者 guozhonggen (编号: 19, 标签: 0): EC: 1窗口, EO: 1窗口
  受试者 hanli (编号: 99, 标签: 0): EC: 1窗口, EO: 1窗口
  受试者 hanyulan (编号: 57, 标签: 1): EC: 1窗口, EO: 1窗口
  受试者 haoshaojin (编号: 25, 标签: 0):

## 保存特征和标签


In [14]:
# 保存三个数据集的特征和标签
# 为每个受试者保存一个特征文件，文件名格式为 feature_017.npy
for dataset_type in ['EC', 'EO', 'All']:
    if len(all_data[dataset_type]) == 0:
        print(f"\n{dataset_type}数据集: 没有数据，跳过保存")
        continue
    
    # 获取输出目录
    output_feature_dir = os.path.join(output_ec_dir if dataset_type == 'EC' else output_eo_dir if dataset_type == 'EO' else output_all_dir, "Feature")
    output_label_dir = os.path.join(output_ec_dir if dataset_type == 'EC' else output_eo_dir if dataset_type == 'EO' else output_all_dir, "Label")
    
    print(f"\n{dataset_type}数据集:")
    
    # 保存每个受试者的特征文件
    labels_list = []
    subject_numbers_list = []
    
    # 按受试者编号排序
    sorted_subject_numbers = sorted(all_data[dataset_type].keys())
    
    for subject_number in sorted_subject_numbers:
        subject_data = all_data[dataset_type][subject_number]
        features = subject_data['features']
        label = subject_data['label']
        subject_name = subject_data['subject_name']
        
        # 保存特征文件，文件名格式为 feature_017.npy（3位数字，前面补0）
        feature_filename = f"feature_{subject_number:03d}.npy"
        feature_path = os.path.join(output_feature_dir, feature_filename)
        np.save(feature_path, features)
        
        # 记录标签信息（每个受试者一个标签）
        labels_list.append(label)
        subject_numbers_list.append(subject_number)
        
        print(f"  受试者 {subject_name} (编号: {subject_number:03d}): {feature_filename}, 形状: {features.shape}, 标签: {label}")
    
    # 保存标签文件
    # 标签文件格式：(受试者数量, 2)，第一列是标签，第二列是对应的受试者编号
    n_subjects = len(labels_list)
    labels_array = np.zeros((n_subjects, 2), dtype=np.int32)
    labels_array[:, 0] = np.array(labels_list)  # 第一列：标签
    labels_array[:, 1] = np.array(subject_numbers_list)  # 第二列：受试者编号
    
    label_path = os.path.join(output_label_dir, "label.npy")
    np.save(label_path, labels_array)
    
    print(f"\n  标签已保存到: {label_path}")
    print(f"  标签形状: {labels_array.shape} (受试者数量, 2)")
    print(f"  标签统计:")
    print(f"    正常人 (NC, 0): {np.sum(labels_array[:, 0] == 0)} 个受试者")
    print(f"    MCI患者 (MCI, 1): {np.sum(labels_array[:, 0] == 1)} 个受试者")
    print(f"    AD患者 (AD, 2): {np.sum(labels_array[:, 0] == 2)} 个受试者")
    print(f"    受试者总数: {n_subjects} 个")


EC数据集:
  受试者 tianxiaolong (编号: 001): feature_001.npy, 形状: (1, 23040, 20), 标签: 1
  受试者 wangyan (编号: 002): feature_002.npy, 形状: (1, 23040, 20), 标签: 1
  受试者 yuandonghui (编号: 003): feature_003.npy, 形状: (1, 23040, 20), 标签: 0
  受试者 wanglijia (编号: 004): feature_004.npy, 形状: (1, 23040, 20), 标签: 1
  受试者 wangjun (编号: 005): feature_005.npy, 形状: (1, 23040, 20), 标签: 0
  受试者 wanghui (编号: 006): feature_006.npy, 形状: (1, 23040, 20), 标签: 0
  受试者 yaolianfen (编号: 007): feature_007.npy, 形状: (1, 23040, 20), 标签: 1
  受试者 liuguifu (编号: 008): feature_008.npy, 形状: (1, 23040, 20), 标签: 0
  受试者 zhangxinhua (编号: 009): feature_009.npy, 形状: (1, 23040, 20), 标签: 0
  受试者 gedeyun (编号: 010): feature_010.npy, 形状: (1, 23040, 20), 标签: 2
  受试者 wuderong (编号: 011): feature_011.npy, 形状: (1, 23040, 20), 标签: 0
  受试者 tianxiangrong (编号: 012): feature_012.npy, 形状: (1, 23040, 20), 标签: 0
  受试者 dengqing (编号: 013): feature_013.npy, 形状: (1, 23040, 20), 标签: 0
  受试者 renwenying (编号: 014): feature_014.npy, 形状: (1, 23040, 20), 标签: 2
  受试者 duji

## 验证数据


In [15]:
# 验证保存的数据
print("\n" + "="*60)
print("验证保存的数据")
print("="*60)

for dataset_type in ['EC', 'EO', 'All']:
    output_feature_dir = os.path.join(output_ec_dir if dataset_type == 'EC' else output_eo_dir if dataset_type == 'EO' else output_all_dir, "Feature")
    output_label_dir = os.path.join(output_ec_dir if dataset_type == 'EC' else output_eo_dir if dataset_type == 'EO' else output_all_dir, "Label")
    
    label_path = os.path.join(output_label_dir, "label.npy")
    
    print(f"\n{dataset_type}数据集:")
    if os.path.exists(label_path):
        loaded_labels = np.load(label_path)
        
        # 获取所有特征文件
        feature_files = [f for f in os.listdir(output_feature_dir) if f.startswith('feature_') and f.endswith('.npy')]
        feature_files.sort()
        
        print(f"  特征文件数量: {len(feature_files)}")
        print(f"  标签文件形状: {loaded_labels.shape} (受试者数量, 2)")
        print(f"  标签文件格式: 第一列=标签, 第二列=受试者编号")
        
        # 验证标签
        print(f"  验证:")
        print(f"    第一列（标签）范围: [{loaded_labels[:, 0].min()}, {loaded_labels[:, 0].max()}] (0=NC, 1=MCI, 2=AD)")
        print(f"    第二列（受试者编号）范围: [{loaded_labels[:, 1].min()}, {loaded_labels[:, 1].max()}]")
        
        # 构建受试者编号到标签的映射
        subject_to_label = {int(row[1]): int(row[0]) for row in loaded_labels}
        
        # 显示前几个特征文件的信息
        print(f"\n  前5个特征文件的信息:")
        for i, feature_file in enumerate(feature_files[:5]):
            feature_path = os.path.join(output_feature_dir, feature_file)
            if os.path.exists(feature_path):
                features = np.load(feature_path)
                # 从文件名中提取受试者编号（例如 feature_001.npy -> 1）
                try:
                    subject_number = int(feature_file.replace('feature_', '').replace('.npy', ''))
                    label_value = subject_to_label.get(subject_number, 'N/A')
                    print(f"    {feature_file}: 形状={features.shape}, 标签={label_value}, 受试者编号={subject_number}")
                except ValueError:
                    print(f"    {feature_file}: 形状={features.shape}, 无法提取受试者编号")
        
        # 显示后几个特征文件的信息
        print(f"\n  后5个特征文件的信息:")
        for feature_file in feature_files[-5:]:
            feature_path = os.path.join(output_feature_dir, feature_file)
            if os.path.exists(feature_path):
                features = np.load(feature_path)
                # 从文件名中提取受试者编号
                try:
                    subject_number = int(feature_file.replace('feature_', '').replace('.npy', ''))
                    label_value = subject_to_label.get(subject_number, 'N/A')
                    print(f"    {feature_file}: 形状={features.shape}, 标签={label_value}, 受试者编号={subject_number}")
                except ValueError:
                    print(f"    {feature_file}: 形状={features.shape}, 无法提取受试者编号")
    else:
        print(f"  警告: 标签文件不存在")



验证保存的数据

EC数据集:
  特征文件数量: 111
  标签文件形状: (111, 2) (受试者数量, 2)
  标签文件格式: 第一列=标签, 第二列=受试者编号
  验证:
    第一列（标签）范围: [0, 2] (0=NC, 1=MCI, 2=AD)
    第二列（受试者编号）范围: [1, 111]

  前5个特征文件的信息:
    feature_001.npy: 形状=(1, 23040, 20), 标签=1, 受试者编号=1
    feature_002.npy: 形状=(1, 23040, 20), 标签=1, 受试者编号=2
    feature_003.npy: 形状=(1, 23040, 20), 标签=0, 受试者编号=3
    feature_004.npy: 形状=(1, 23040, 20), 标签=1, 受试者编号=4
    feature_005.npy: 形状=(1, 23040, 20), 标签=0, 受试者编号=5

  后5个特征文件的信息:
    feature_107.npy: 形状=(1, 23040, 20), 标签=0, 受试者编号=107
    feature_108.npy: 形状=(1, 23040, 20), 标签=0, 受试者编号=108
    feature_109.npy: 形状=(1, 23040, 20), 标签=0, 受试者编号=109
    feature_110.npy: 形状=(1, 23040, 20), 标签=0, 受试者编号=110
    feature_111.npy: 形状=(1, 23040, 20), 标签=0, 受试者编号=111

EO数据集:
  特征文件数量: 111
  标签文件形状: (111, 2) (受试者数量, 2)
  标签文件格式: 第一列=标签, 第二列=受试者编号
  验证:
    第一列（标签）范围: [0, 2] (0=NC, 1=MCI, 2=AD)
    第二列（受试者编号）范围: [1, 111]

  前5个特征文件的信息:
    feature_001.npy: 形状=(1, 23040, 20), 标签=1, 受试者编号=1
    feature_002.npy: 形状=(1, 2304

In [16]:
# 测试加载数据文件
print("\n" + "="*60)
print("测试加载数据文件")
print("="*60)

def count_original_samples(shape):
    """增强 layout (2, N, T, C) 时只统计第 1 维原始样本数 N"""
    if len(shape) == 4 and shape[0] == 2:
        return shape[1]
    return shape[0]

for dataset_type in ['EC', 'EO', 'All']:
    output_feature_dir = os.path.join(output_ec_dir if dataset_type == 'EC' else output_eo_dir if dataset_type == 'EO' else output_all_dir, "Feature")
    output_label_dir = os.path.join(output_ec_dir if dataset_type == 'EC' else output_eo_dir if dataset_type == 'EO' else output_all_dir, "Label")
    
    label_path = os.path.join(output_label_dir, "label.npy")
    
    print(f"\n{dataset_type}数据集:")
    if os.path.exists(label_path):
        test_labels = np.load(label_path)
        
        # 获取所有特征文件
        feature_files = [f for f in os.listdir(output_feature_dir) if f.startswith('feature_') and f.endswith('.npy')]
        feature_files.sort()
        
        if len(feature_files) > 0:
            # 测试加载第一个和最后一个特征文件
            first_feature_path = os.path.join(output_feature_dir, feature_files[0])
            last_feature_path = os.path.join(output_feature_dir, feature_files[-1])
            
            if os.path.exists(first_feature_path) and os.path.exists(last_feature_path):
                first_features = np.load(first_feature_path)
                last_features = np.load(last_feature_path)
                
                print(f"  特征文件数量: {len(feature_files)}")
                print(f"  标签文件形状: {test_labels.shape}")
                print(f"  第一个特征文件 ({feature_files[0]}): 形状={first_features.shape}")
                print(f"  最后一个特征文件 ({feature_files[-1]}): 形状={last_features.shape}")

                if dataset_type == 'EO':
                    eo_total_samples = 0
                    for fname in feature_files:
                        fpath = os.path.join(output_feature_dir, fname)
                        eo_total_samples += count_original_samples(np.load(fpath).shape)
                    print(f"  原始样本总数: {eo_total_samples}")

                print(f"  数据加载成功！")
            else:
                print(f"  警告: 特征文件不存在")
        else:
            print(f"  警告: 没有找到特征文件")
    else:
        print(f"  警告: 标签文件不存在")


测试加载数据文件

EC数据集:
  特征文件数量: 111
  标签文件形状: (111, 2)
  第一个特征文件 (feature_001.npy): 形状=(1, 23040, 20)
  最后一个特征文件 (feature_111.npy): 形状=(1, 23040, 20)
  数据加载成功！

EO数据集:
  特征文件数量: 111
  标签文件形状: (111, 2)
  第一个特征文件 (feature_001.npy): 形状=(1, 23040, 20)
  最后一个特征文件 (feature_111.npy): 形状=(1, 23040, 20)
  原始样本总数: 111
  数据加载成功！

All数据集:
  特征文件数量: 111
  标签文件形状: (111, 2)
  第一个特征文件 (feature_001.npy): 形状=(2, 23040, 20)
  最后一个特征文件 (feature_111.npy): 形状=(2, 23040, 20)
  数据加载成功！


### 获取 CAEEG3_1.0s_0.0_EO_Aug 数据集中每个受试者的样本数

In [17]:
# 获取 CAEEG3_1.0s_0.0_EO_Aug 数据集中每个受试者的样本数并保存
print("\n" + "=" * 60)
print("统计 CAEEG3_1.0s_0.0_EO_Aug 各受试者样本数")
print("=" * 60)

dataset_name = "CAEEG3_1.0s_0.0_EO_Aug"
eo_output_base_dir = r"/root/autodl-tmp/ADMamba/dataset"
dataset_dir = os.path.join(eo_output_base_dir, dataset_name)
output_feature_dir = os.path.join(dataset_dir, "Feature")
output_label_dir = os.path.join(dataset_dir, "Label")
label_path = os.path.join(output_label_dir, "label.npy")


def count_original_samples(shape):
    """增强 layout (2, N, T, C) 时只统计第 1 维原始样本数 N"""
    if len(shape) == 4 and shape[0] == 2:
        return shape[1]
    return shape[0]


if not os.path.exists(label_path):
    raise FileNotFoundError(f"标签文件不存在: {label_path}")

loaded_labels = np.load(label_path)
subject_sample_counts = []
missing_features = []

for label, subject_id in loaded_labels:
    subject_id = int(subject_id)
    label = int(label)
    feature_file = f"feature_{subject_id:03d}.npy"
    feature_path = os.path.join(output_feature_dir, feature_file)

    if os.path.exists(feature_path):
        features = np.load(feature_path)
        n_samples = count_original_samples(features.shape)
        subject_sample_counts.append([label, subject_id, n_samples])
    else:
        missing_features.append(subject_id)
        subject_sample_counts.append([label, subject_id, 0])

subject_sample_counts = np.array(subject_sample_counts, dtype=np.int64)

# 保存到 autodl-tmp/ADMamba/dataset/ 目录
npy_save_path = os.path.join(eo_output_base_dir, f"{dataset_name}_subject_sample_counts.npy")
np.save(npy_save_path, subject_sample_counts)

print(f"数据集路径: {dataset_dir}")
print(f"受试者数量: {len(subject_sample_counts)}")
print(f"样本数范围: [{subject_sample_counts[:, 2].min()}, {subject_sample_counts[:, 2].max()}]")
print(f"样本总数: {subject_sample_counts[:, 2].sum()}")
print(f"\n已保存: {npy_save_path}")
print(f"  格式: 第一列=标签(0=NC,1=MCI,2=AD), 第二列=受试者编号, 第三列=样本数")

if missing_features:
    print(f"\n警告: 以下受试者缺少特征文件: {missing_features}")

print(f"\n前5个受试者:")
for label, subject_id, n_samples in subject_sample_counts[:5]:
    print(f"  feature_{subject_id:03d}.npy: 标签={label}, 样本数={n_samples}")

print(f"\n后5个受试者:")
for label, subject_id, n_samples in subject_sample_counts[-5:]:
    print(f"  feature_{subject_id:03d}.npy: 标签={label}, 样本数={n_samples}")



统计 CAEEG3_1.0s_0.0_EO_Aug 各受试者样本数
数据集路径: /root/autodl-tmp/ADMamba/dataset/CAEEG3_1.0s_0.0_EO_Aug
受试者数量: 111
样本数范围: [147, 232]
样本总数: 19699

已保存: /root/autodl-tmp/ADMamba/dataset/CAEEG3_1.0s_0.0_EO_Aug_subject_sample_counts.npy
  格式: 第一列=标签(0=NC,1=MCI,2=AD), 第二列=受试者编号, 第三列=样本数

前5个受试者:
  feature_001.npy: 标签=1, 样本数=178
  feature_002.npy: 标签=1, 样本数=178
  feature_003.npy: 标签=0, 样本数=178
  feature_004.npy: 标签=1, 样本数=178
  feature_005.npy: 标签=0, 样本数=178

后5个受试者:
  feature_107.npy: 标签=0, 样本数=177
  feature_108.npy: 标签=0, 样本数=178
  feature_109.npy: 标签=0, 样本数=178
  feature_110.npy: 标签=0, 样本数=171
  feature_111.npy: 标签=0, 样本数=178
